# PathSignal footpath-damage candidate (research only)

Fine-tune an image segmentation model on the Indian Footpath Damage Segmentation Dataset. This is an offline candidate evaluation, not an app integration or a safety claim. It detects annotated visible footpath damage; it does not measure depth or distance.

The dataset contains 583 images with pixel masks and severity metadata (411 train / 85 validation / 87 test). The source archive is 4.5 GB. Kaggle Internet and GPU are required. Zenodo record 15414238 declares CC BY 4.0. Cite Chakurkar et al. and retain attribution.

Run all cells from top to bottom. Select the checkpoint using validation IoU, then evaluate the held-out test once. This is not an unseen-city evaluation.

In [ ]:
# Kaggle Settings: Internet ON, Accelerator GPU T4.
import json, random, shutil, urllib.request, zipfile
from pathlib import Path
import numpy as np, torch
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Settings > Accelerator.'
print('GPU:',torch.cuda.get_device_name(0))
URL='https://zenodo.org/api/records/15414238/files/Indian%20Footpath%20Damage%20Segmentation%20Dataset.zip/content'
ZIP=Path('/kaggle/working/footpath-damage.zip'); DATA=Path('/kaggle/working/footpath-damage-data'); OUT=Path('/kaggle/working/pathsignal-footpath-candidate'); OUT.mkdir(exist_ok=True)
if not ZIP.exists() or ZIP.stat().st_size < 4_000_000_000: print('Downloading 4.5 GB...'); urllib.request.urlretrieve(URL,ZIP)
assert ZIP.stat().st_size > 4_000_000_000, 'Dataset download is incomplete.'
if not DATA.exists():
    with zipfile.ZipFile(ZIP) as z:
        bad=z.testzip(); assert bad is None, f'Corrupt archive member: {bad}'
        z.extractall(DATA)
files=[p for p in DATA.rglob('*') if p.is_file()]
print('Extracted files:',len(files)); print('Examples:',[str(p.relative_to(DATA)) for p in files[:30]])

In [ ]:
# Pair images and binary damage masks while retaining the published train/val/test folders.
from PIL import Image
from collections import Counter
IMG_EXT={'.jpg','.jpeg','.bmp','.tif','.tiff'}; MASK_EXT={'.png','.bmp','.tif','.tiff'}
def ismask(p):
    s=(p.stem+' '+str(p.parent)).lower()
    return p.suffix.lower() in MASK_EXT and any(k in s for k in ('mask','label','annotation','ground_truth','ground-truth','segmentation'))
def key(p):
    s=p.stem.lower()
    for t in ('_mask','-mask',' mask','_label','-label','_annotation','-annotation','_segmentation','-segmentation','_gt','-gt'): s=s.replace(t,'')
    return s
masks=[p for p in files if ismask(p)]; images=[p for p in files if p.suffix.lower() in IMG_EXT and not ismask(p)]
mb={}; [mb.setdefault(key(p),[]).append(p) for p in masks]
pairs=[]
for im in images:
    cand=mb.get(key(im),[])
    if not cand: continue
    parts=[x.lower() for x in im.relative_to(DATA).parts]
    split=next((s for s in ('train','val','test') if s in parts),None)
    if split: pairs.append((split,im,cand[0]))
counts=Counter(x[0] for x in pairs); print('Paired counts:',dict(counts),'unpaired images:',len(images)-len(pairs),'masks:',len(masks))
assert counts.get('train',0)>=300 and counts.get('val',0)>=50 and counts.get('test',0)>=50, 'Automatic image-mask pairing failed. Inspect the printed examples and adjust the pairing logic before training.'
PREP=Path('/kaggle/working/footpath-prepared')
if PREP.exists(): shutil.rmtree(PREP)
for split,im,ma in pairs:
    (PREP/split/'images').mkdir(parents=True,exist_ok=True); (PREP/split/'masks').mkdir(parents=True,exist_ok=True)
    name=key(im)
    Image.open(im).convert('RGB').save(PREP/split/'images'/(name+'.jpg'),quality=95)
    Image.open(ma).convert('L').save(PREP/split/'masks'/(name+'.png'))
print({s:len(list((PREP/s/'images').glob('*.jpg'))) for s in ('train','val','test')})

In [ ]:
# Transfer-learn a binary semantic segmenter. Test split remains untouched until the next cell.
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.models import ResNet50_Weights
SIZE=512; BATCH=6; EPOCHS=12; DEVICE='cuda'
class FootpathDS(Dataset):
    def __init__(self,split): self.split=split; self.ims=sorted((PREP/split/'images').glob('*.jpg'))
    def __len__(self): return len(self.ims)
    def __getitem__(self,i):
        ip=self.ims[i]; mp=PREP/self.split/'masks'/(ip.stem+'.png')
        im=Image.open(ip).convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        ma=Image.open(mp).convert('L').resize((SIZE,SIZE),Image.Resampling.NEAREST)
        x=np.asarray(im,dtype=np.float32)/255; x=(x-np.array([.485,.456,.406],np.float32))/np.array([.229,.224,.225],np.float32)
        y=(np.asarray(ma,dtype=np.uint8)>0).astype(np.int64)
        return torch.from_numpy(x.transpose(2,0,1)),torch.from_numpy(y)
train_ds=FootpathDS('train'); val_ds=FootpathDS('val'); test_ds=FootpathDS('test')
train_dl=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True); val_dl=DataLoader(val_ds,batch_size=BATCH,num_workers=2,pin_memory=True); test_dl=DataLoader(test_ds,batch_size=BATCH,num_workers=2,pin_memory=True)
model=deeplabv3_resnet50(weights=None,weights_backbone=ResNet50_Weights.DEFAULT,num_classes=2).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4); scaler=torch.amp.GradScaler('cuda')
def run_epoch(dl,training=False):
    model.train(training); inter=union=tp=fp=fn=0; loss_sum=0; batches=0
    for x,y in dl:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        with torch.set_grad_enabled(training),torch.autocast('cuda',dtype=torch.float16):
            logits=model(x)['out']; logits=F.interpolate(logits,size=y.shape[-2:],mode='bilinear',align_corners=False)
            onehot=F.one_hot(y,num_classes=2).permute(0,3,1,2).float(); prob=logits.softmax(1)
            bce=F.cross_entropy(logits,y); dice=1-(2*(prob*onehot).sum((0,2,3))+1)/(prob.sum((0,2,3))+onehot.sum((0,2,3))+1); loss=bce+dice[1]
        if training: opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        pred=logits.argmax(1)==1; gt=y==1; inter+=int((pred&gt).sum()); union+=int((pred|gt).sum()); tp+=int((pred&gt).sum()); fp+=int((pred&~gt).sum()); fn+=int((~pred&gt).sum()); loss_sum+=float(loss.detach()); batches+=1
    return {'loss':loss_sum/max(batches,1),'foreground_iou':inter/max(union,1),'precision':tp/max(tp+fp,1),'recall':tp/max(tp+fn,1)}
best=-1; bestfile=OUT/'best-deeplabv3-resnet50.pth'; history=[]; stale=0
for epoch in range(1,EPOCHS+1):
    tr=run_epoch(train_dl,True); va=run_epoch(val_dl); row={'epoch':epoch,'train':tr,'validation':va}; history.append(row); print(row)
    if va['foreground_iou']>best: best=va['foreground_iou']; stale=0; torch.save(model.state_dict(),bestfile)
    else: stale+=1
    if stale>=3: print('Early stop by validation IoU'); break
(OUT/'training-history.json').write_text(json.dumps(history,indent=2)); print('Selected checkpoint:',bestfile,'val IoU:',best)

In [ ]:
# Load the validation-selected checkpoint, evaluate held-out test once, and export.
model.load_state_dict(torch.load(bestfile,map_location=DEVICE,weights_only=True))
test_metrics=run_epoch(test_dl)
report={'dataset':'Indian Footpath Damage Segmentation Dataset','doi':'10.5281/zenodo.15414238','license':'CC BY 4.0','task':'visible footpath damage segmentation','test_images':len(test_ds),'selected_by':'validation foreground IoU','test_metrics':test_metrics,'limitations':['Source locations are in Pune, India, not a held-out South African site.','Damage masks do not establish trip risk, depth, or warning distance.','This candidate is not integrated into PathSignal.']}
(OUT/'evaluation-report.json').write_text(json.dumps(report,indent=2))
class Export(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,x): return self.m(x)['out'].softmax(1)[:,1:2]
torch.onnx.export(Export(model.eval()).cpu(),torch.zeros(1,3,SIZE,SIZE),str(OUT/'footpath-damage-deeplabv3.onnx'),input_names=['images'],output_names=['damage_probability'],dynamic_axes={'images':{0:'batch'},'damage_probability':{0:'batch'}},opset_version=17)
print(json.dumps(report,indent=2)); print('Artifacts:',[p.name for p in OUT.iterdir()])

## Interpreting the output
Report held-out foreground IoU, precision, and recall from `evaluation-report.json`, not training accuracy. This is an offline segmentation candidate; it is not a production-ready alert model. It needs local damaged/intact-site evaluation, on-device decoder/runtime checks, and alert timing before app integration.

Citation: Chakurkar, P. et al. (2025), *Indian Footpath Damage Segmentation Dataset*, Zenodo, https://doi.org/10.5281/zenodo.15414238. CC BY 4.0; credit the source and note adaptations.